<a href="https://colab.research.google.com/github/microsoft/qlib/blob/main/examples/workflow_by_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
sys.path.insert(0, r"D:\gitdesktop\Qtrade\qlib")

import qlib
print(qlib.__file__)

import pandas as pd
from qlib.constant import REG_CN
from qlib.utils import exists_qlib_data, init_instance_by_config
from qlib.workflow import R
from qlib.workflow.record_temp import SignalRecord, PortAnaRecord
from qlib.utils import flatten_dict
from qlib.tests.data import GetData

/home/shengwang/miniconda3/envs/qlib/lib/python3.13/site-packages/qlib/__init__.py


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
provider_uri = r"/home/shengwang/Documents/qlib/qlib_bin_norm"  # target_dir
qlib.init(provider_uri=provider_uri, region=REG_CN)

[12038:MainThread](2025-09-07 11:33:19,147) INFO - qlib.Initialization - [config.py:451] - default_conf: client.
[12038:MainThread](2025-09-07 11:33:19,151) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.
[12038:MainThread](2025-09-07 11:33:19,152) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/home/shengwang/Documents/qlib/qlib_bin_norm')}


In [3]:
market = "csi1000"
benchmark = "SH000300"

# train model

In [4]:
import datetime

# 获取今天日期
today = datetime.date.today()
today_str = today.strftime("%Y-%m-%d")
print("今天：" + today_str)

# 获取昨天的日期
yesterday = today - datetime.timedelta(days=1)
yesterday_str = yesterday.strftime("%Y-%m-%d")
print("昨天：" + yesterday_str)

# # 格式化为字符串
# end_time = yesterday.strftime("%Y-%m-%d")

start_time_str = "2023-01-01"
start_time = datetime.datetime.strptime(start_time_str, "%Y-%m-%d")
start_time_yesterday = start_time - datetime.timedelta(days=1)
start_time_yesterday_str = start_time_yesterday.strftime("%Y-%m-%d")
end_time = today_str

print("回测开始时间："+start_time_str)
print("回测结束时间："+end_time)

今天：2025-09-07
昨天：2025-09-06
回测开始时间：2023-01-01
回测结束时间：2025-09-07


In [5]:
###################################
# train model
###################################
data_handler_config = {
    "start_time": "2008-01-01",
    "end_time": end_time,
    "fit_start_time": "2008-01-01",
    "fit_end_time": "2020-12-31",                   # 用来这段时间控制的是 数据预处理时，用来计算“全局参数”的时间范围，比如训练数据中所有特征的均值和方差。
    "instruments": market,
}

task = {
    "dataset": {
        "class": "DatasetH",
        "module_path": "qlib.data.dataset",
        "kwargs": {
            "handler": {
                "class": "Alpha158",  #158 / 360
                "module_path": "qlib.contrib.data.handler",
                "kwargs": data_handler_config,
            },
            "segments": {
                "train": ("2008-01-01", "2020-12-31"),           # 训练模型的样本
                "valid": ("2021-01-01", start_time_yesterday_str),           # 调参或早停用的验证集
                "test": (start_time_str, end_time),            # 回测阶段评估模型表现的数据集
            },
        },
    },
}

dataset = init_instance_by_config(task["dataset"])

# 获取数据索引范围（通常是 datetime index）
df = dataset.handler.fetch(col_set="feature")  # 或 col_set="all" 也行
# 查看数据的时间范围
print("数据开始日期：", df.index.min())
print("数据结束日期：", df.index.max())
print("数据的 features 列：", df.columns.tolist())

[12038:MainThread](2025-09-07 11:34:05,643) INFO - qlib.timer - [log.py:127] - Time cost: 46.473s | Loading data Done
[12038:MainThread](2025-09-07 11:34:06,392) INFO - qlib.timer - [log.py:127] - Time cost: 0.310s | DropnaLabel Done
[12038:MainThread](2025-09-07 11:34:07,645) INFO - qlib.timer - [log.py:127] - Time cost: 1.252s | CSZScoreNorm Done
[12038:MainThread](2025-09-07 11:34:07,649) INFO - qlib.timer - [log.py:127] - Time cost: 2.006s | fit & process data Done
[12038:MainThread](2025-09-07 11:34:07,650) INFO - qlib.timer - [log.py:127] - Time cost: 48.481s | Init data Done


数据开始日期： (Timestamp('2014-10-31 00:00:00'), 'SH600680')
数据结束日期： (Timestamp('2025-09-05 00:00:00'), 'SZ301631')
数据的 features 列： ['KMID', 'KLEN', 'KMID2', 'KUP', 'KUP2', 'KLOW', 'KLOW2', 'KSFT', 'KSFT2', 'OPEN0', 'HIGH0', 'LOW0', 'VWAP0', 'ROC5', 'ROC10', 'ROC20', 'ROC30', 'ROC60', 'MA5', 'MA10', 'MA20', 'MA30', 'MA60', 'STD5', 'STD10', 'STD20', 'STD30', 'STD60', 'BETA5', 'BETA10', 'BETA20', 'BETA30', 'BETA60', 'RSQR5', 'RSQR10', 'RSQR20', 'RSQR30', 'RSQR60', 'RESI5', 'RESI10', 'RESI20', 'RESI30', 'RESI60', 'MAX5', 'MAX10', 'MAX20', 'MAX30', 'MAX60', 'MIN5', 'MIN10', 'MIN20', 'MIN30', 'MIN60', 'QTLU5', 'QTLU10', 'QTLU20', 'QTLU30', 'QTLU60', 'QTLD5', 'QTLD10', 'QTLD20', 'QTLD30', 'QTLD60', 'RANK5', 'RANK10', 'RANK20', 'RANK30', 'RANK60', 'RSV5', 'RSV10', 'RSV20', 'RSV30', 'RSV60', 'IMAX5', 'IMAX10', 'IMAX20', 'IMAX30', 'IMAX60', 'IMIN5', 'IMIN10', 'IMIN20', 'IMIN30', 'IMIN60', 'IMXD5', 'IMXD10', 'IMXD20', 'IMXD30', 'IMXD60', 'CORR5', 'CORR10', 'CORR20', 'CORR30', 'CORR60', 'CORD5', 'CORD1

In [6]:
# 1. 统计每只股票的最早日期
first_dates = df.groupby(df.index.get_level_values(1)).apply(
    lambda x: x.index.get_level_values(0).min()
)

# 2. 变成DataFrame
first_dates_df = first_dates.reset_index()
first_dates_df.columns = ['股票代码', '入市时间']

# 3. 按入市时间从晚到早排序
first_dates_df = first_dates_df.sort_values('入市时间', ascending=False)

# 4. 打印最晚入市的股票
most_recent_stock = first_dates_df.iloc[0]
print("最晚入市的股票:")
print(f"股票代码：{most_recent_stock['股票代码']}")
print(f"入市时间：{most_recent_stock['入市时间']}")

# 5. 打印全部列表
print("股票入市时间排序列表:")
print(first_dates_df)

# 或者更美观地打印
for idx, row in first_dates_df.iterrows():
    print(f"{row['股票代码']}\t{row['入市时间']}")

最晚入市的股票:
股票代码：SZ301631
入市时间：2025-06-30 00:00:00
股票入市时间排序列表:
          股票代码       入市时间
2686  SZ301631 2025-06-30
1666  SZ002368 2025-06-30
662   SH603193 2025-06-30
663   SH603194 2025-06-30
1049  SH688584 2025-06-30
...        ...        ...
1872  SZ002636 2015-05-29
1277  SZ000789 2015-05-29
328   SH600680 2014-10-31
1398  SZ001914 2014-10-31
1396  SZ001872 2014-10-31

[2687 rows x 2 columns]
SZ301631	2025-06-30 00:00:00
SZ002368	2025-06-30 00:00:00
SH603193	2025-06-30 00:00:00
SH603194	2025-06-30 00:00:00
SH688584	2025-06-30 00:00:00
SH688165	2025-06-30 00:00:00
SZ001356	2025-06-30 00:00:00
SH603395	2025-06-30 00:00:00
SH603444	2025-06-30 00:00:00
SZ001339	2025-06-30 00:00:00
SH603516	2025-06-30 00:00:00
SH688567	2025-06-30 00:00:00
SH603667	2025-06-30 00:00:00
SH603698	2025-06-30 00:00:00
SZ002408	2025-06-30 00:00:00
SH603883	2025-06-30 00:00:00
SZ300925	2025-06-30 00:00:00
SZ301458	2025-06-30 00:00:00
SH688543	2025-06-30 00:00:00
SH688536	2025-06-30 00:00:00
SZ300972	2025-06-30 00: